# Text-to-SVG Generation: Inference & Evaluation Pipeline

**Model:** LoRA-adapted Qwen2.5-Coder-3B-Instruct 
**Task:** Generate SVG code from prompts, post-process, evaluate, and submit

## 1. Environment & Imports

In [ ]:
import os
import re
import gc
import torch
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

import cv2
import cairosvg
from io import BytesIO
from PIL import Image
from zss import simple_distance, Node
from skimage.metrics import structural_similarity as ssim
from IPython.display import display, SVG, HTML
from tqdm.auto import tqdm

## 2. Model Loading

Load the base Qwen2.5-Coder-3B model and attach the trained LoRA adapter. The adapter is merged into the base weights for faster inference.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"
ADAPTER_DIR = "./outputs_long_ver12"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, local_files_only=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Attaching Trained LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, local_files_only=True)
model = model.merge_and_unload()
model.eval()

print("✅ Model ready for pure PyTorch inference!")

## 3. SVG Validation Utilities

These functions enforce the Kaggle competition constraints: allowed tags, character limit, path count limit, and renderability.

In [ ]:
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline', 
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask', 'linearGradient', 
    'radialGradient', 'stop', 'text', 'tspan', 'title', 'desc', 'style', 
    'pattern', 'marker', 'filter'
}

def check_validity_with_reason(svg_str):
    if not svg_str: return 0, "Empty string generated."
    if len(svg_str) > 16000: return 0, f"Exceeds max length constraint (Length: {len(svg_str)} > 16000)."
    if svg_str.count("<path") > 256: return 0, f"Exceeds max path count constraint (Paths: {svg_str.count('<path')} > 256)."
    
    try:
        root = ET.fromstring(svg_str)
        for elem in root.iter():
            tag = elem.tag.split('}')[-1]
            if tag not in ALLOWED_TAGS:
                return 0, f"Contains disallowed XML tag: <{tag}>"
    except ET.ParseError as e:
        return 0, f"XML Parse Error: {e}"
        
    return 1, "✅ Valid"

def is_blank_svg(svg_str):
    """
    Checks if the SVG actually contains any drawable geometric shapes.
    Returns True if it's just an empty canvas, False if it has content.
    """
    if not svg_str:
        return True
    
    # List of all standard SVG shape tags
    shape_tags = ['<path', '<rect', '<circle', '<ellipse', '<polygon', '<polyline', '<line']
    
    # If NONE of these tags are in the string, it's a blank image.
    if not any(tag in svg_str for tag in shape_tags):
        return True
        
    return False

def render_svg_to_rgb_array(svg_str, name="SVG"):
    try:
        if not isinstance(svg_str, str):
            print(f"⚠️ {name} is not a string!")
            return None
            
        png_data = cairosvg.svg2png(bytestring=svg_str.encode('utf-8'), output_width=256, output_height=256)
        # CHANGED: Convert to RGB, not 'L' (Grayscale)
        img = Image.open(BytesIO(png_data)).convert('RGB')
        return np.array(img)
    except Exception as e:
        # print(f"⚠️ CairoSVG specifically failed on {name} because: {e}")
        return None

def calc_edge_f1(img_pred_rgb, img_ref_rgb):
    # CHANGED: Convert RGB to Grayscale specifically for the Edge detector
    gray_pred = cv2.cvtColor(img_pred_rgb, cv2.COLOR_RGB2GRAY)
    gray_ref = cv2.cvtColor(img_ref_rgb, cv2.COLOR_RGB2GRAY)
    
    edges_pred = cv2.Canny(gray_pred, 50, 150) > 0
    edges_ref = cv2.Canny(gray_ref, 50, 150) > 0
    
    tp = np.logical_and(edges_pred, edges_ref).sum()
    fp = np.logical_and(edges_pred, ~edges_ref).sum()
    fn = np.logical_and(~edges_pred, edges_ref).sum()
    
    if tp == 0: return 0.0
    return 2 * tp / (2 * tp + fp + fn)

def xml_to_tree(elem):
    node = Node(elem.tag.split('}')[-1])
    for child in elem: node.addkid(xml_to_tree(child))
    return node

## 4. Post-Processing Pipeline

Our post-processing pipeline addresses common generation failures:

1. **Fuzzy Stutter Guillotine** — Detects and truncates repetitive path commands (e.g., the model gets stuck repeating `L128 32` hundreds of times). Uses both exact pattern matching and fuzzy tolerance-based detection.

2. **Tag Healer** — Fixes SVGs that were cut off mid-generation due to token limits. Closes unclosed path `d` attributes, balances `<g>` tags, and appends `</svg>`.

3. **Exact Deduplication** — Removes duplicate path elements (same `d` string with same fill/stroke). Also filters zero-area ghost paths that render as invisible points.

4. **CairoSVG Hotfixes** — Fixes known rendering issues: empty fills, `currentColor` replacement, arc flag normalization.

**Ablation note:** Without the stutter guillotine, approximately 10-15% of outputs contain severe repetition that fills the entire token budget with garbage. The deduplication step catches additional duplicates that the stutter guillotine misses (whole-element duplicates vs within-path duplicates).

In [ ]:
# 1. UPDATED SYSTEM PROMPT: Forces the model to use compact repetition!
SYSTEM_PROMPT = (
    "You are an expert vector graphics designer"
    "Generate highly compact, parseable, and valid SVG code"
    "Focus on the dominant colors and bounding boxes of the requested objects"
)


import re
import torch
import xml.etree.ElementTree as ET

# ==============================================================================
# HELPER 1: THE FUZZY STUTTER GUILLOTINE (Text-Level)
# Runs on raw text. Detects loops (with tolerance) and chops the string!
# ==============================================================================
def fuzzy_stutter_guillotine(d_str, tolerance=2.0, max_repeats=3):
    # 1. Poly-Command Number Stutter (e.g., repeating " 0-1.3 0-2.6")
    poly_stutter_match = re.search(r'(.{10,100}?)\1{4,}', d_str)
    if poly_stutter_match:
        d_str = d_str[:poly_stutter_match.end(1)] # Chop it immediately!

    # 2. Fuzzy Command Stutter (e.g., repeating "C 100.1 200" and "C 101.4 199")
    chunks = [c for c in re.split(r'(?=[a-zA-Z])', d_str) if c.strip()]
    cleaned_chunks = []
    consecutive_stutters = 0
    
    def is_fuzzy_match(chunk1, chunk2):
        if re.findall(r'[a-zA-Z]', chunk1) != re.findall(r'[a-zA-Z]', chunk2): return False
        nums1 = [float(x) for x in re.findall(r'-?\d+(?:\.\d+)?', chunk1)]
        nums2 = [float(x) for x in re.findall(r'-?\d+(?:\.\d+)?', chunk2)]
        if len(nums1) != len(nums2): return False
        for a, b in zip(nums1, nums2):
            if abs(a - b) > tolerance: return False
        return True

    for chunk in chunks:
        if len(cleaned_chunks) > 0 and is_fuzzy_match(chunk, cleaned_chunks[-1]):
            consecutive_stutters += 1
            if consecutive_stutters >= max_repeats:
                break # GUILLOTINE DROPS! Stop reading.
        else:
            consecutive_stutters = 0
            cleaned_chunks.append(chunk)
            
    return "".join(cleaned_chunks)

# ==============================================================================
# HELPER 2: EXACT XML DEDUPLICATION & CAIROSVG HOTFIXES
# Runs on parsed XML. Deletes EXACT duplicates and fixes rendering bugs.
# ==============================================================================
def armor_and_deduplicate_svg(svg_str):
    try:
        clean_str = re.sub(r'\sxmlns=[\'"][^\'"]+[\'"]', '', svg_str)
        root = ET.fromstring(clean_str)
        
        # We store EXACT tuples: (exact_d_string, is_filled, is_stroked)
        seen_path_signatures = set()
        
        for parent in root.iter():
            to_remove = []
            for child in parent:
                tag = child.tag.split('}')[-1]
                
                if tag == 'path' and 'd' in child.attrib:
                    d_string = child.attrib['d'].strip()
                    
                    # 1. Zero-Area Ghost Path Filter (Deletes invisible garbage)
                    nums = [float(n) for n in re.findall(r'-?\d+(?:\.\d+)?', d_string)]
                    if nums:
                        x_coords = nums[0::2]
                        y_coords = nums[1::2] if len(nums) > 1 else [nums[0]]
                        if (max(x_coords) - min(x_coords) < 0.1) and (max(y_coords) - min(y_coords) < 0.1):
                            to_remove.append(child)
                            continue 
                    
                    # 2. STRICT EXACT DEDUPLICATION (Aware of Fills vs Strokes!)
                    fill_attr = child.get('fill') or parent.get('fill', 'none')
                    stroke_attr = child.get('stroke') or parent.get('stroke', 'none')
                    
                    is_filled = (fill_attr.lower() not in ['none', ''])
                    is_stroked = (stroke_attr.lower() not in ['none', ''])
                    
                    # Signature uses the EXACT string, no rounding here!
                    signature = (d_string, is_filled, is_stroked)
                    
                    if signature in seen_path_signatures:
                        to_remove.append(child) # It's a Black Blob/Loop! Delete it.
                    else:
                        seen_path_signatures.add(signature)
            
            for child in to_remove:
                parent.remove(child)
                
        inner = ET.tostring(root, encoding='unicode', method='xml')
        inner_content_match = re.search(r'<svg[^>]*>(.*?)</svg>', inner, flags=re.IGNORECASE | re.DOTALL)
        
        if inner_content_match:
            inner_content = inner_content_match.group(1).strip()
            
            # 3. FORCE KAGGLE HEADER
            final_svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">{inner_content}</svg>'
            
            # 4. FIX CAIROSVG COLOR & INVISIBLE SHAPE CRASHES
            final_svg = final_svg.replace('currentColor', '#333333')
            # Fix invisible white-on-white
            final_svg = re.sub(r'fill="#[Ff]{6}"', 'fill="#EEEEEE"', final_svg)
            final_svg = re.sub(r'fill="#[Ff]{3}"', 'fill="#EEE"', final_svg)
            final_svg = re.sub(r'fill="white"', 'fill="#EEEEEE"', final_svg)
            final_svg = re.sub(r'stroke="#[Ff]{6}"', 'stroke="#CCCCCC"', final_svg)
            final_svg = re.sub(r'stroke="#[Ff]{3}"', 'stroke="#CCC"', final_svg)
            final_svg = re.sub(r'stroke="white"', 'stroke="#CCCCCC"', final_svg)
            final_svg = re.sub(r'\s*filling="[^"]*"', '', final_svg)
            
            if 'stroke=' not in final_svg and 'stroke:' not in final_svg:
                # No stroke? fill="" makes it invisible. Force it to be visible!
                final_svg = final_svg.replace('fill=""', 'fill="#333333"')
            else:
                # Has a stroke? Safe to make the fill transparent.
                final_svg = final_svg.replace('fill=""', 'fill="none"')
                
            # 5. FIX ARC FLAG CRASHES (Forces sweep/large-arc flags to strictly 0 or 1)
            def arc_replacer(m):
                p1 = m.group(1)
                f1 = '1' if float(m.group(2)) >= 0.5 else '0'
                f2 = '1' if float(m.group(3)) >= 0.5 else '0'
                return f"{p1} {f1} {f2}"
            
            final_svg = re.sub(r'([Aa]\s*[\d\.\-]+\s+[\d\.\-]+\s+[\d\.\-]+)\s+([\d\.\-]+)\s+([\d\.\-]+)', arc_replacer, final_svg)
            
            # 6. Stop off-screen drawing
            final_svg = re.sub(r'<g transform="scale\([^)]+\)">', '<g>', final_svg)
            
            return final_svg
        return svg_str
    except Exception:
        return svg_str

# ==============================================================================
# MAIN INFERENCE FUNCTION
# ==============================================================================
def generate_svg(prompt, max_new_tokens=4096):
    FORCED_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    
    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{FORCED_HEADER}"
    )
    
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,            # GREEDY DECODING
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[
                tokenizer.eos_token_id, 
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
                tokenizer.encode("</svg>", add_special_tokens=False)[-1]
            ]
        )
        
    prompt_length = inputs['input_ids'].shape[1]
    decoded = tokenizer.decode(output_ids[0][prompt_length:], skip_special_tokens=True)
    decoded = FORCED_HEADER + decoded
    print("raw svg:\n")
    print(decoded)

    # ========================================================
    # 1. RUN THE FUZZY STUTTER GUILLOTINE ON RAW TEXT
    # ========================================================
    def replacer(match):
        d_content = match.group(1)
        quote = match.group(2)
        cleaned_content = fuzzy_stutter_guillotine(d_content, tolerance=2.0)
        return f'd="{cleaned_content}{quote}'
        
    decoded = re.sub(r'd="([^"]*)("?)', replacer, decoded)

    # ========================================================
    # 2. THE "ANY-CUT" SMART TAG HEALER
    # ========================================================
    if "</svg>" not in decoded:
        last_open = decoded.rfind('<')
        last_close = decoded.rfind('>')
        
        if last_open > last_close: 
            unclosed_text = decoded[last_open:]
            if unclosed_text.startswith('<path') and 'd="' in unclosed_text:
                last_space = unclosed_text.rfind(' ')
                if last_space > unclosed_text.find('d="'):
                    decoded = decoded[:last_open + last_space] + '" />'
                else:
                    decoded = decoded + '" />' 
            else:
                decoded = decoded[:last_open] 
                
        open_g_count = decoded.count('<g') - decoded.count('</g>')
        decoded += ("</g>" * max(0, open_g_count)) + "</svg>"
        
    # ========================================================
    # 3. EXTRACT AND APPLY STRICT EXACT DEDUPLICATION
    # ========================================================
    svg_match = re.search(r"<svg[\s\S]*?</svg>", decoded, flags=re.IGNORECASE)
    final_svg = svg_match.group(0).strip() if svg_match else ""
    
    if final_svg:
        final_svg = armor_and_deduplicate_svg(final_svg)
        
    return final_svg

## 5. Local Evaluation (Optional)

Score generated SVGs against ground-truth references using the same metrics as the Kaggle competition:
- **Visual Fidelity (V_i, weight 0.85):** 0.7 × SSIM + 0.3 × Edge F1
- **Structural Similarity (S_i, weight 0.12):** exp(-TED/25)
- **Compactness (C_i, weight 0.03):** exp(-|log((len_pred+50)/(len_ref+50))|)

This section is for local debugging only — uncomment to run against your eval split.

In [ ]:
# import signal
# from datasets import Dataset

# # --- Define a custom timeout exception and handler ---
# class TimeoutException(BaseException):
#     pass

# def timeout_handler(signum, frame):
#     raise TimeoutException()

# # Attach the handler to the system alarm signal
# signal.signal(signal.SIGALRM, timeout_handler)
# # -----------------------------------------------------

# # --- CONFIGURATION ---
# SEED = 42
# EVAL_SIZE = 0.02 
# SAMPLE_SIZE = 20 # Set this to a higher number when doing your real evaluation
# # ---------------------

# print("Loading eval dataset")
# eval_ds = Dataset.from_pandas(pd.read_csv("eval.csv"))

# # 3. Randomly sample prompts strictly from the eval split
# eval_subset = eval_ds.shuffle(seed=SEED).select(range(SAMPLE_SIZE))

# print(f"Testing {SAMPLE_SIZE} unseen prompts...\n")

# scores =[]
# invalid_count = 0

# for idx, row in enumerate(eval_subset):
#     prompt = row['prompt']
#     ref_svg = row['svg']
    
#     print("="*60)
#     print(f"🎨 PROMPT {idx + 1}/{SAMPLE_SIZE}: {prompt}")
#     print("="*60)
    
#     # 1. Generate
#     pred_svg = generate_svg(prompt, max_new_tokens=2048)
#     print('Processed SVG:\n')
#     print(pred_svg)

#     # 2. Stage 1: Validity Gate
#     is_valid, reason = check_validity_with_reason(pred_svg)
#     print(f"Status: {reason}")
    
#     if is_valid == 0:
#         scores.append(0.0)
#         invalid_count += 1
#         print("Score for this sample: 0.0\n")
#         print("--- RAW SVG TEXT (FAILED VALIDITY) ---")
#         print(pred_svg)
#         print("--------------------------------------\n\n")
#         continue 
        
#     # --- START TIMEOUT CLOCK (30 seconds) ---
#     try:
#         signal.alarm(30) # Give the math 30 seconds to finish
        
#         # 3. Stage 2: Quality Score
#         img_pred = render_svg_to_rgb_array(pred_svg, name="Generated SVG")
#         img_ref = render_svg_to_rgb_array(ref_svg, name="Ground Truth SVG")
        
#         if img_pred is None:
#             scores.append(0.0)
#             invalid_count += 1
#             print("❌ Render Error: Generated SVG failed to render.")
#             signal.alarm(0)
#             continue
            
#         if img_ref is None:
#             # If the Kaggle dataset's ground truth is broken, we shouldn't punish the model!
#             # But we skip it because we can't do the SSIM math without a reference image.
#             print("⚠️ Ground Truth SVG is broken! Skipping this sample so it doesn't ruin the score.")
#             signal.alarm(0)
#             continue

#         # Visual Fidelity
#         ssim_val = ssim(img_pred, img_ref, data_range=255, channel_axis=-1)
#         edge_f1 = calc_edge_f1(img_pred, img_ref)
#         V_i = (0.7 * ssim_val) + (0.3 * edge_f1)
        
#         # Structural Similarity (Tree Edit Distance)
#         try:
#             pred_root = xml_to_tree(ET.fromstring(pred_svg))
#             ref_root = xml_to_tree(ET.fromstring(ref_svg))
#             ted = simple_distance(pred_root, ref_root)
#         except:
#             ted = 25 
#         S_i = np.exp(-ted / 25.0)
        
#         # Compactness
#         C_i = np.exp(-abs(np.log((len(pred_svg) + 50) / (len(ref_svg) + 50))))
        
#         # Composite Score
#         Q_i = (V_i ** 0.85) * (S_i ** 0.12) * (C_i ** 0.03)
#         scores.append(Q_i)
        
#         print(f"📊 SAMPLE METRICS -> V_i: {V_i:.3f} | S_i: {S_i:.3f} | C_i: {C_i:.3f}")
#         print(f"Sample Score (Q_i): {Q_i:.4f}")
        
#         print("\nGenerated SVG Output:")
#         display(SVG(pred_svg)) # Successfully display the image!
#         print("\n\n")
        
#         signal.alarm(0) # Success! Stop the clock.
        
#     except TimeoutException:
#         # If it takes more than 30 seconds, it gets trapped here
#         print("⏳ TIMEOUT: The generated SVG was too complex and froze the scorer! Scoring 0.0")
#         scores.append(0.0)
#         invalid_count += 1
#         print("--- RAW SVG TEXT (TIMEOUT) ---")
#         print(pred_svg)
#         print("------------------------------\n\n")
        
#     except Exception as e:
#         # If it crashes for any other weird mathematical reason
#         print(f"⚠️ UNEXPECTED SCORING ERROR: {e}")
#         scores.append(0.0)
#         invalid_count += 1
#         print("--- RAW SVG TEXT (UNEXPECTED ERROR) ---")
#         print(pred_svg)
#         print("---------------------------------------\n\n")
        
#     finally:
#         # Always make sure the alarm is turned off before the next loop
#         signal.alarm(0) 

# # ==========================================
# # FINAL LEADERBOARD SCORE CALCULATION
# # ==========================================
# final_leaderboard_score = 100 * np.mean(scores)

# print("#"*60)
# print("🏆 OVERALL EVALUATION COMPLETE 🏆")
# print("#"*60)
# print(f"Total Samples Tested : {SAMPLE_SIZE}")
# print(f"Valid SVGs Generated : {SAMPLE_SIZE - invalid_count}")
# print(f"Invalid/Failed SVGs  : {invalid_count} (Scored 0.0)")
# print(f"\n🌟 FINAL ESTIMATED LEADERBOARD SCORE: {final_leaderboard_score:.4f} / 100 🌟")
# print("#"*60)

## 6. Batch Submission (Greedy + Retry)

The baseline submission pipeline:
1. **Pass 1 (Greedy):** Generate one SVG per prompt with `do_sample=False`
2. **Pass 2 (Sampling Rescue):** For prompts that failed validation in Pass 1, retry with `temperature=0.4` sampling

In [ ]:
import pandas as pd
import torch
import re
from tqdm.auto import tqdm

# --- 0. PRE-FLIGHT SETUP ---

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 1. FALLBACK SVG ---
def fallback_svg():
    return '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="64" fill="#888888"/></svg>'

# --- 2. BATCH GENERATOR (TRUE GREEDY DECODING) ---
def generate_svg_batch_submission(prompts, max_new_tokens=4096, do_sample=False, temperature=0.15, top_p=0.95):
    FORCED_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">\n'
    
    chat_texts = []
    for p in prompts:
        chat_text = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{p}<|im_end|>\n"
            f"<|im_start|>assistant\n{FORCED_HEADER}"
        )
        chat_texts.append(chat_text)
        
    inputs = tokenizer(chat_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,           # <--- NOW DYNAMIC
            temperature=temperature if do_sample else None, # Only pass temp if sampling
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[
                tokenizer.eos_token_id, 
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
                tokenizer.encode("</svg>", add_special_tokens=False)[-1]
            ]
        )
        
    prompt_length = inputs['input_ids'].shape[1]
    decoded_batch = tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True)
    
    results = []
    for decoded in decoded_batch:
        decoded = FORCED_HEADER + decoded

        # 1. RUN THE FUZZY STUTTER GUILLOTINE ON RAW TEXT
        def replacer(match):
            d_content = match.group(1)
            quote = match.group(2)
            cleaned_content = fuzzy_stutter_guillotine(d_content, tolerance=2.0)
            return f'd="{cleaned_content}{quote}'
            
        decoded = re.sub(r'd="([^"]*)("?)', replacer, decoded)

        # 2. THE "ANY-CUT" SMART TAG HEALER
        if "</svg>" not in decoded:
            last_open = decoded.rfind('<')
            last_close = decoded.rfind('>')
            
            if last_open > last_close: 
                unclosed_text = decoded[last_open:]
                if unclosed_text.startswith('<path') and 'd="' in unclosed_text:
                    last_space = unclosed_text.rfind(' ')
                    if last_space > unclosed_text.find('d="'):
                        decoded = decoded[:last_open + last_space] + '" />'
                    else:
                        decoded = decoded + '" />' 
                else:
                    decoded = decoded[:last_open] 
                    
            open_g_count = decoded.count('<g') - decoded.count('</g>')
            decoded += ("</g>" * max(0, open_g_count)) + "</svg>"
            
        # 3. EXTRACT AND APPLY STRICT EXACT DEDUPLICATION
        svg_match = re.search(r"<svg[\s\S]*?</svg>", decoded, flags=re.IGNORECASE)
        final_svg = svg_match.group(0).strip() if svg_match else ""
        
        if final_svg:
            final_svg = armor_and_deduplicate_svg(final_svg)
            
        results.append(final_svg)
        
    return results


# # ==============================================================================
# # MAIN KAGGLE SUBMISSION LOOP (TWO-PASS RETRY SYSTEM)
# # ==============================================================================
TEST_CSV_PATH = "test.csv"
SUBMISSION_PATH = "./outputs_long_ver15/submission.csv"
BATCH_SIZE = 64 

print(f"Loading {TEST_CSV_PATH}...")
try:
    test_df = pd.read_csv(TEST_CSV_PATH)
except FileNotFoundError:
    test_df = pd.DataFrame({"id": ["dummy_1", "dummy_2"], "prompt":["a circle", "a square"]})

if not test_df.empty:
    # Use a dictionary to store results so we can easily overwrite failures
    final_results = {}
    
    failed_prompts = []
    failed_ids = []
    
    print(f"--- PASS 1: GREEDY GENERATION ({len(test_df)} prompts) ---")
    
    for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Pass 1 (Greedy)"):
        batch_df = test_df.iloc[i:i+BATCH_SIZE]
        prompts = batch_df['prompt'].tolist()
        ids = batch_df['id'].tolist()
        
        # GENERATE GREEDY
        pred_svgs = generate_svg_batch_submission(prompts, max_new_tokens=2048, do_sample=False)
        
        for j, svg_str in enumerate(pred_svgs):
            current_id = ids[j]
            current_prompt = prompts[j]
            
            is_valid, reason = check_validity_with_reason(svg_str)
            is_blank = is_blank_svg(svg_str)
            
            if is_valid == 1 and not is_blank: 
                final_results[current_id] = svg_str
            else:
                # IT FAILED! Save it for Pass 2 instead of instantly giving up.
                failed_prompts.append(current_prompt)
                failed_ids.append(current_id)
                final_results[current_id] = None # Placeholder

    # ==========================================
    # PASS 2: THE SAMPLING RESCUE MISSION
    # ==========================================
    ultimate_fail_count = 0
    
    if failed_prompts:
        print(f"\n--- PASS 2: RESCUING {len(failed_prompts)} FAILED SVGS WITH SAMPLING ---")
        
        for i in tqdm(range(0, len(failed_prompts), BATCH_SIZE), desc="Pass 2 (Sampling)"):
            batch_prompts = failed_prompts[i:i+BATCH_SIZE]
            batch_ids = failed_ids[i:i+BATCH_SIZE]
            
            # GENERATE WITH SAMPLING (Higher temp to shake it out of the trap)
            rescue_svgs = generate_svg_batch_submission(
                batch_prompts, 
                max_new_tokens=2048, 
                do_sample=True, 
                temperature=0.4,  # 0.4 provides enough creativity to break hard structural traps
                top_p=0.95,
            )
            
            for j, rescue_svg in enumerate(rescue_svgs):
                current_id = batch_ids[j]
                
                is_valid, reason = check_validity_with_reason(rescue_svg)
                is_blank = is_blank_svg(rescue_svg)
            
                if is_valid == 1 and not is_blank: 
                    final_results[current_id] = svg_str
                else:
                    # If it STILL fails after sampling, we finally surrender to the dummy square
                    final_results[current_id] = fallback_svg()
                    ultimate_fail_count += 1
    
    # Format the dictionary back into a DataFrame for submission
    submission_rows = [{"id": k, "svg": v} for k, v in final_results.items()]
    submission_df = pd.DataFrame(submission_rows)
    
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    
    print("\n" + "="*50)
    print("✅ SUBMISSION GENERATION COMPLETE")
    print("="*50)
    print(f"Total Processed   : {len(submission_df)}")
    print(f"Rescued in Pass 2 : {len(failed_prompts) - ultimate_fail_count}")
    print(f"Ultimate Failures : {ultimate_fail_count} (Replaced with blank SVG)")
    print(f"Saved exactly to  : {SUBMISSION_PATH}")

## 7. Best-of-3 Sampling (Ablation)

An alternative submission strategy that generates 3 candidates per prompt (1 greedy + 2 sampled) and selects the best by pixel variance after rendering.

**Selection metric:** `np.var(rendered_image)` — higher variance indicates more visual content, which correlates with better SSIM against non-trivial references. A blank white canvas has variance ≈0; a colorful icon has variance >2000.

**Ablation note:** This approach trades 3× inference time (~6 hours vs ~2 hours) for potentially better output selection. Every prompt benefits from diversity, not just the ones that fail validation.

**Runtime:** ~5 hours on RTX 4080 Super for 1000 prompts.

In [ ]:
# """
# BEST-OF-3 — Drop-in replacement for your submission loop cell
# ==============================================================
# Replaces the two-pass greedy+sampling system.
# Generates 1 greedy + 2 sampled per prompt, picks best by pixel variance.
# """

# import pandas as pd
# import numpy as np
# import torch
# import re
# from tqdm.auto import tqdm

# # --- PRE-FLIGHT ---
# tokenizer.padding_side = "left"
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# # --- FALLBACK ---
# def fallback_svg():
#     return '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="64" fill="#888888"/></svg>'

# # --- RENDER FOR SCORING ---
# def render_to_array(svg_str):
#     try:
#         png_data = cairosvg.svg2png(bytestring=svg_str.encode('utf-8'),
#                                      output_width=256, output_height=256)
#         img = Image.open(BytesIO(png_data)).convert('RGB')
#         return np.array(img)
#     except Exception:
#         return None

# # --- PICK BEST CANDIDATE ---
# def pick_best(candidates):
#     """Pick best SVG from candidates by pixel variance."""
#     best_svg = None
#     best_var = -1

#     for svg in candidates:
#         if not svg:
#             continue
#         is_valid, _ = check_validity_with_reason(svg)
#         if is_valid == 0:
#             continue
#         if is_blank_svg(svg):
#             continue
#         arr = render_to_array(svg)
#         if arr is None:
#             continue
#         v = np.var(arr.astype(float))
#         if v > best_var:
#             best_var = v
#             best_svg = svg

#     return best_svg

# # --- BATCH GENERATOR (unchanged from your original) ---
# def generate_svg_batch(prompts, max_new_tokens=2048, do_sample=False, temperature=0.4, top_p=0.95):
#     FORCED_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">\n'

#     chat_texts = []
#     for p in prompts:
#         chat_text = (
#             f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
#             f"<|im_start|>user\n{p}<|im_end|>\n"
#             f"<|im_start|>assistant\n{FORCED_HEADER}"
#         )
#         chat_texts.append(chat_text)

#     inputs = tokenizer(chat_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=do_sample,
#             temperature=temperature if do_sample else None,
#             top_p=top_p if do_sample else None,
#             pad_token_id=tokenizer.pad_token_id,
#             eos_token_id=[
#                 tokenizer.eos_token_id,
#                 tokenizer.convert_tokens_to_ids("<|im_end|>"),
#                 tokenizer.encode("</svg>", add_special_tokens=False)[-1]
#             ]
#         )

#     prompt_length = inputs['input_ids'].shape[1]
#     decoded_batch = tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True)

#     results = []
#     for decoded in decoded_batch:
#         decoded = FORCED_HEADER + decoded

#         # Strip filling
#         decoded = re.sub(r'\s*filling="[^"]*"', '', decoded)

#         # Stutter guillotine
#         def replacer(match):
#             d_content = match.group(1)
#             quote = match.group(2)
#             cleaned_content = fuzzy_stutter_guillotine(d_content, tolerance=2.0)
#             return f'd="{cleaned_content}{quote}'
#         decoded = re.sub(r'd="([^"]*)("?)', replacer, decoded)

#         # Tag healer
#         if "</svg>" not in decoded:
#             last_open = decoded.rfind('<')
#             last_close = decoded.rfind('>')
#             if last_open > last_close:
#                 unclosed_text = decoded[last_open:]
#                 if unclosed_text.startswith('<path') and 'd="' in unclosed_text:
#                     last_space = unclosed_text.rfind(' ')
#                     if last_space > unclosed_text.find('d="'):
#                         decoded = decoded[:last_open + last_space] + '" />'
#                     else:
#                         decoded = decoded + '" />'
#                 else:
#                     decoded = decoded[:last_open]
#             open_g_count = decoded.count('<g') - decoded.count('</g>')
#             decoded += ("</g>" * max(0, open_g_count)) + "</svg>"

#         # Extract and deduplicate
#         svg_match = re.search(r"<svg[\s\S]*?</svg>", decoded, flags=re.IGNORECASE)
#         final_svg = svg_match.group(0).strip() if svg_match else ""
#         if final_svg:
#             final_svg = armor_and_deduplicate_svg(final_svg)

#         results.append(final_svg)

#     return results


# # =============================================================================
# # MAIN SUBMISSION LOOP — BEST OF 3
# # =============================================================================
# TEST_CSV_PATH = "test.csv"
# SUBMISSION_PATH = "./outputs_long_ver12/submission_ver2.csv"
# BATCH_SIZE = 32  # Smaller than 64 since we do 3 passes

# print(f"Loading {TEST_CSV_PATH}...")
# try:
#     test_df = pd.read_csv(TEST_CSV_PATH)
# except FileNotFoundError:
#     test_df = pd.DataFrame({"id": ["dummy_1", "dummy_2"], "prompt": ["a circle", "a square"]})

# if not test_df.empty:
#     final_results = {}
#     fallback_count = 0

#     print(f"--- BEST-OF-3: {len(test_df)} prompts ---")

#     for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Generating"):
#         batch_df = test_df.iloc[i:i + BATCH_SIZE]
#         prompts = batch_df['prompt'].tolist()
#         ids = batch_df['id'].tolist()

#         # Pass 1: Greedy
#         greedy_svgs = generate_svg_batch(prompts, max_new_tokens=2048, do_sample=False)

#         # Pass 2: Sampling run 1
#         sample1_svgs = generate_svg_batch(prompts, max_new_tokens=2048, do_sample=True, temperature=0.4, top_p=0.95)

#         # Pass 3: Sampling run 2
#         sample2_svgs = generate_svg_batch(prompts, max_new_tokens=2048, do_sample=True, temperature=0.4, top_p=0.95)

#         # Pick best for each prompt
#         for j in range(len(prompts)):
#             candidates = [greedy_svgs[j], sample1_svgs[j], sample2_svgs[j]]
#             best = pick_best(candidates)

#             if best is not None:
#                 final_results[ids[j]] = best
#             else:
#                 final_results[ids[j]] = fallback_svg()
#                 fallback_count += 1

#     # Save
#     submission_rows = [{"id": k, "svg": v} for k, v in final_results.items()]
#     submission_df = pd.DataFrame(submission_rows)
#     submission_df.to_csv(SUBMISSION_PATH, index=False)

#     print(f"\n{'='*50}")
#     print("SUBMISSION COMPLETE")
#     print(f"{'='*50}")
#     print(f"Total:          {len(submission_df)}")
#     print(f"Fallbacks used: {fallback_count}")
#     print(f"Saved to:       {SUBMISSION_PATH}")